In [6]:
# timeline_model.py
"""
BiLSTM Timeline Context Model
==============================
Processes an entire timeline of post embeddings through a BiLSTM to produce
context-aware embeddings, then passes them through two staged classifiers.

Architecture
------------
  post_embeddings (T, D)
         ↓
  [Optional Input Projection]  D → proj_dim
         ↓
  BiLSTM (num_layers, bidirectional)
         ↓
  context_embeddings (T, 2H)   ← one per post, full temporal context
         ↓
  ┌──────────────────────────────────────────────────────────────────────┐
  │  STAGE 1 — TaxonomyClassifier                                        │
  │  12 × CrossEntropy heads  (one per valence × dimension)              │
  │    adaptive  : A(8), B-O(3), B-S(2), C-O(3), C-S(2), D(4)          │
  │    maladaptive: A(8), B-O(3), B-S(2), C-O(3), C-S(2), D(4)         │
  │  2 × BCE heads  (Switch, Escalation)  — threshold 0.5               │
  └──────────────────────────────────────────────────────────────────────┘
         ↓  (freeze encoder + classifier1)
  ┌──────────────────────────────────────────────────────────────────────┐
  │  STAGE 2 — PresenceRegressor  (RMSE loss)                           │
  │    adaptive_presence  (1-5)                                          │
  │    maladaptive_presence (1-5)                                        │
  └──────────────────────────────────────────────────────────────────────┘

Usage
-----
  # Stage 1
  trainer = TimelineModelTrainer(model, device=device)
  trainer.train_stage1(train_loader, val_loader, epochs=15, save_path="s1.pt")

  # Stage 2
  model.load_state_dict(torch.load("s1.pt"))
  trainer.train_stage2(train_loader, val_loader, epochs=10, save_path="s2.pt")

  # Inference
  infer = TimelineInference(model, device=device)
  preds = infer.predict(timeline_embeddings)   # list of per-post dicts
  ctx   = infer.get_context_embeddings(timeline_embeddings)  # (T, 2H)
"""

from __future__ import annotations

import math
import os
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

from data_structure import Post, Timeline

# ─────────────────────────────────────────────────────────────────────────────
# 1.  Label definitions
# ─────────────────────────────────────────────────────────────────────────────

DIMENSIONS = ["A", "B-O", "B-S", "C-O", "C-S", "D"]

# Per-head LOCAL class indices (None = "not present in this dimension")
# These are the targets passed to CrossEntropyLoss for each head.
HEAD_LABEL_MAP: Dict[str, Dict[str, Dict]] = {
    "adaptive": {
        "A":   {1: 0, 3: 1, 5: 2,  7: 3,  9: 4,  11: 5, 13: 6, None: 7},
        "B-O": {1: 0, 3: 1, None: 2},
        "B-S": {1: 0, None: 1},
        "C-O": {1: 0, 3: 1, None: 2},
        "C-S": {1: 0, None: 1},
        "D":   {1: 0, 3: 1, 5: 2,  None: 3},
    },
    "maladaptive": {
        "A":   {2: 0, 4: 1, 6: 2,  8: 3,  10: 4, 12: 5, 14: 6, None: 7},
        "B-O": {2: 0, 4: 1, None: 2},
        "B-S": {2: 0, None: 1},
        "C-O": {2: 0, 4: 1, None: 2},
        "C-S": {2: 0, None: 1},
        "D":   {2: 0, 4: 1, 6: 2,  None: 3},
    },
}

# Number of output classes per head
HEAD_SIZES: Dict[str, Dict[str, int]] = {
    valence: {dim: max(idx_map.values()) + 1 for dim, idx_map in dim_map.items()}
    for valence, dim_map in HEAD_LABEL_MAP.items()
}

# Canonical ordered list of (valence, dim) pairs → 12 heads total
HEAD_ORDER: List[Tuple[str, str]] = [
    (v, d) for v in ["adaptive", "maladaptive"] for d in DIMENSIONS
]

PRESENCE_MIN = 1.0
PRESENCE_MAX = 5.0


# ─────────────────────────────────────────────────────────────────────────────
# 2.  Label conversion helpers
# ─────────────────────────────────────────────────────────────────────────────

def _post_to_head_labels(post: Post) -> np.ndarray:
    """
    Returns (12,) int64 array of per-head local class indices.
    
    For each of the 12 (valence, dimension) pairs:
      - If the post's self-state has a subelement for that dimension,
        return its local index.
      - Otherwise return the 'None' class index (last class of each head).
    """
    labels = np.empty(12, dtype=np.int64)
    for h_idx, (valence, dim) in enumerate(HEAD_ORDER):
        state   = post.adaptive_state if valence == "adaptive" else post.maladaptive_state
        idx_map = HEAD_LABEL_MAP[valence][dim]
        none_idx = idx_map[None]
        se_map  = state.by_dimension   # {dim_str: SubElement}
        if dim in se_map:
            labels[h_idx] = idx_map.get(se_map[dim].number, none_idx)
        else:
            labels[h_idx] = none_idx
    return labels


# ─────────────────────────────────────────────────────────────────────────────
# 3.  Dataset
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class TimelineSample:
    """One fully processed timeline ready for the model."""
    timeline_id:       str
    embeddings:        np.ndarray   # (T, D)  float32
    head_labels:       np.ndarray   # (T, 12) int64   – per-head local targets
    switch_labels:     np.ndarray   # (T,)    float32 – binary
    escalation_labels: np.ndarray   # (T,)    float32 – binary
    ada_presence:      np.ndarray   # (T,)    float32 – 1.0 … 5.0
    mal_presence:      np.ndarray   # (T,)    float32 – 1.0 … 5.0
    is_annotated:      np.ndarray   # (T,)    bool    – loss mask


class TimelineDataset(Dataset):
    """
    Each sample is one complete Timeline.
    The full post embedding sequence forms the input; padded in collate.

    Parameters
    ----------
    timelines          : list of Timeline objects (posts must have post_embedding set)
    skip_no_embedding  : drop timelines where any post has None embedding
    """

    def __init__(
        self,
        timelines:         List[Timeline],
        skip_no_embedding: bool = True,
    ) -> None:
        self._samples: List[TimelineSample] = []
        skipped = 0

        for tl in timelines:
            posts = tl.posts
            if not posts:
                skipped += 1
                continue
            if skip_no_embedding and any(p.post_embedding is None for p in posts):
                skipped += 1
                continue

            embeddings        = np.stack([p.post_embedding for p in posts], axis=0).astype(np.float32)
            head_labels       = np.stack([_post_to_head_labels(p) for p in posts], axis=0)
            switch_labels     = np.array([1.0 if p.is_switch     else 0.0 for p in posts], dtype=np.float32)
            escalation_labels = np.array([1.0 if p.is_escalation else 0.0 for p in posts], dtype=np.float32)
            ada_presence      = np.array([float(p.adaptive_state.presence)    for p in posts], dtype=np.float32)
            mal_presence      = np.array([float(p.maladaptive_state.presence) for p in posts], dtype=np.float32)
            is_annotated      = np.array([p.is_annotated for p in posts], dtype=bool)

            self._samples.append(TimelineSample(
                timeline_id       = tl.timeline_id,
                embeddings        = embeddings,
                head_labels       = head_labels,
                switch_labels     = switch_labels,
                escalation_labels = escalation_labels,
                ada_presence      = ada_presence,
                mal_presence      = mal_presence,
                is_annotated      = is_annotated,
            ))

        print(f"[TimelineDataset] {len(self._samples)} timelines loaded  ({skipped} skipped).")

    def __len__(self) -> int:
        return len(self._samples)

    def __getitem__(self, idx: int) -> TimelineSample:
        return self._samples[idx]

    @staticmethod
    def collate(batch: List[TimelineSample]) -> Dict:
        """
        Pad variable-length timelines to the longest in the batch.

        Returns
        -------
        embeddings        : (B, T_max, D)   – padded with zeros
        lengths           : List[int]        – real length per sample
        head_labels       : (B, T_max, 12)  – padded with 0 (ignored via ann_mask)
        switch_labels     : (B, T_max)
        escalation_labels : (B, T_max)
        ada_presence      : (B, T_max)      – padded with 1.0
        mal_presence      : (B, T_max)      – padded with 1.0
        ann_mask          : (B, T_max) bool – True = annotated post → compute loss here
        pad_mask          : (B, T_max) bool – True = real post (not padding)
        """
        lengths = [s.embeddings.shape[0] for s in batch]
        T_max   = max(lengths)
        D       = batch[0].embeddings.shape[1]
        B       = len(batch)

        emb_pad  = torch.zeros(B, T_max, D,   dtype=torch.float32)
        hl_pad   = torch.zeros(B, T_max, 12,  dtype=torch.long)
        sw_pad   = torch.zeros(B, T_max,      dtype=torch.float32)
        esc_pad  = torch.zeros(B, T_max,      dtype=torch.float32)
        ada_pad  = torch.ones(B,  T_max,      dtype=torch.float32)
        mal_pad  = torch.ones(B,  T_max,      dtype=torch.float32)
        ann_mask = torch.zeros(B, T_max,      dtype=torch.bool)
        pad_mask = torch.zeros(B, T_max,      dtype=torch.bool)

        for i, s in enumerate(batch):
            T = s.embeddings.shape[0]
            emb_pad[i, :T]  = torch.from_numpy(s.embeddings)
            hl_pad[i, :T]   = torch.from_numpy(s.head_labels)
            sw_pad[i, :T]   = torch.from_numpy(s.switch_labels)
            esc_pad[i, :T]  = torch.from_numpy(s.escalation_labels)
            ada_pad[i, :T]  = torch.from_numpy(s.ada_presence)
            mal_pad[i, :T]  = torch.from_numpy(s.mal_presence)
            ann_mask[i, :T] = torch.from_numpy(s.is_annotated)
            pad_mask[i, :T] = True

        return {
            "embeddings":        emb_pad,
            "lengths":           lengths,
            "head_labels":       hl_pad,
            "switch_labels":     sw_pad,
            "escalation_labels": esc_pad,
            "ada_presence":      ada_pad,
            "mal_presence":      mal_pad,
            "ann_mask":          ann_mask,
            "pad_mask":          pad_mask,
        }


# ─────────────────────────────────────────────────────────────────────────────
# 4.  BiLSTM Context Encoder
# ─────────────────────────────────────────────────────────────────────────────

class BiLSTMContextEncoder(nn.Module):
    """
    Processes a padded sequence of post embeddings and outputs a
    context-aware embedding for every post position.

    Input  : (B, T, input_dim)
    Output : (B, T, 2 × hidden_dim)

    Design
    ------
    - Optional learnable input projection (input_dim → proj_dim) with
      LayerNorm + GELU reduces dimensionality before the LSTM,
      preventing overfitting when embeddings are very high-dimensional.
    - N-layer BiLSTM with inter-layer dropout.
    - LayerNorm on the final output stabilises training.
    - Packed sequences ensure padding tokens never influence hidden states.

    Parameters
    ----------
    input_dim     : dimensionality of post embeddings (e.g. 1634 if using PostEmbedder)
    hidden_dim    : size of each LSTM direction (output dim = 2 × hidden_dim)
    num_layers    : number of stacked BiLSTM layers
    dropout       : dropout between LSTM layers and on output
    proj_dim      : project input to this dim before LSTM (None = no projection)
    """

    def __init__(
        self,
        input_dim:  int,
        hidden_dim: int   = 512,
        num_layers: int   = 2,
        dropout:    float = 0.3,
        proj_dim:   Optional[int] = None,
    ) -> None:
        super().__init__()

        if proj_dim is not None and proj_dim != input_dim:
            self.input_proj: nn.Module = nn.Sequential(
                nn.Linear(input_dim, proj_dim),
                nn.LayerNorm(proj_dim),
                nn.GELU(),
            )
            lstm_in = proj_dim
        else:
            self.input_proj = nn.Identity()
            lstm_in = input_dim

        self.lstm = nn.LSTM(
            input_size    = lstm_in,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout if num_layers > 1 else 0.0,
        )

        self.out_norm  = nn.LayerNorm(2 * hidden_dim)
        self.drop      = nn.Dropout(dropout)
        self.output_dim = 2 * hidden_dim

    def forward(
        self,
        embeddings: torch.Tensor,   # (B, T, D)
        lengths:    List[int],
    ) -> torch.Tensor:              # (B, T, 2H)
        x = self.input_proj(embeddings)                                   # (B, T, lstm_in)
        packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out_packed, _ = self.lstm(packed)
        out, _        = pad_packed_sequence(out_packed, batch_first=True) # (B, T, 2H)
        out = self.out_norm(out)
        out = self.drop(out)
        return out                                                         # (B, T, 2H)


# ─────────────────────────────────────────────────────────────────────────────
# 5.  Classifier 1 — Taxonomy + Switch/Escalation
# ─────────────────────────────────────────────────────────────────────────────

class TaxonomyClassifier(nn.Module):
    """
    12 independent softmax heads (one per valence × dimension) +
    2 sigmoid heads (Switch, Escalation).

    Each head shares a common projection layer then branches into its own
    linear output, keeping parameter count manageable.

    Head sizes
    ----------
    adaptive   A(8)  B-O(3)  B-S(2)  C-O(3)  C-S(2)  D(4)
    maladaptive A(8)  B-O(3)  B-S(2)  C-O(3)  C-S(2)  D(4)
    switch/escalation: single logit (BCEWithLogits)

    Input  : context (B, T, context_dim)
    Output : dict[head_name → (B, T, n_classes)]
             "switch"     → (B, T, 1)
             "escalation" → (B, T, 1)

    Inference (argmax / sigmoid ≥ threshold) maps back to subelement numbers
    via the inverse of HEAD_LABEL_MAP.

    Parameters
    ----------
    context_dim : output dim of BiLSTMContextEncoder (= 2 × lstm_hidden)
    hidden_dim  : shared projection hidden size
    dropout     : applied after the shared projection
    """

    def __init__(
        self,
        context_dim: int,
        hidden_dim:  int   = 256,
        dropout:     float = 0.2,
    ) -> None:
        super().__init__()

        # Shared projection: context_dim → hidden_dim
        self.shared = nn.Sequential(
            nn.Linear(context_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # One linear head per (valence, dimension) pair
        self.dim_heads = nn.ModuleDict({
            f"{valence}_{dim}": nn.Linear(hidden_dim, HEAD_SIZES[valence][dim])
            for valence, dim in HEAD_ORDER
        })

        # Switch and Escalation heads (single logit each)
        self.switch_head     = nn.Linear(hidden_dim, 1)
        self.escalation_head = nn.Linear(hidden_dim, 1)

    def forward(self, context: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        context : (B, T, context_dim)
        returns : {
            "adaptive_A"       : (B, T, 8),
            "adaptive_B-O"     : (B, T, 3),
            ...
            "maladaptive_D"    : (B, T, 4),
            "switch"           : (B, T, 1),
            "escalation"       : (B, T, 1),
        }
        """
        h = self.shared(context)   # (B, T, hidden_dim)

        logits: Dict[str, torch.Tensor] = {}
        for valence, dim in HEAD_ORDER:
            key = f"{valence}_{dim}"
            logits[key] = self.dim_heads[key](h)   # (B, T, n_cls)

        logits["switch"]     = self.switch_head(h)       # (B, T, 1)
        logits["escalation"] = self.escalation_head(h)   # (B, T, 1)

        return logits


# ─────────────────────────────────────────────────────────────────────────────
# 6.  Classifier 2 — Presence Regressor  (Stage 2, frozen backbone)
# ─────────────────────────────────────────────────────────────────────────────

class PresenceRegressor(nn.Module):
    """
    Small MLP mapping the context embedding to adaptive and maladaptive
    presence scores (continuous in [1, 5]).

    Trained in Stage 2 with the encoder and TaxonomyClassifier frozen.
    RMSE loss is used so the penalty scales linearly with prediction error,
    which matches the ordinal 1-5 scale.

    Input  : context (B, T, context_dim)
    Output : (B, T, 2)  — channel 0: ada_presence, channel 1: mal_presence
                          clamped to [1.0, 5.0]

    Parameters
    ----------
    context_dim : matches encoder output_dim
    hidden_dim  : MLP hidden size
    dropout     : regularisation
    """

    def __init__(
        self,
        context_dim: int,
        hidden_dim:  int   = 128,
        dropout:     float = 0.1,
    ) -> None:
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(context_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 2),
        )

    def forward(self, context: torch.Tensor) -> torch.Tensor:
        """
        context : (B, T, context_dim)   (may be a no_grad tensor)
        returns : (B, T, 2)  clamped to [1, 5]
        """
        raw = self.net(context)                       # (B, T, 2)
        return torch.clamp(raw, min=PRESENCE_MIN, max=PRESENCE_MAX)


# ─────────────────────────────────────────────────────────────────────────────
# 7.  Full Model
# ─────────────────────────────────────────────────────────────────────────────

class TimelineModel(nn.Module):
    """
    Wires encoder → classifier1 (stage 1) and encoder → classifier2 (stage 2).

    Stage 1 forward : encoder + classifier1
    Stage 2 forward : encoder (frozen, no_grad) + classifier2

    Parameters
    ----------
    input_dim    : dimensionality of post embeddings from PostEmbedder
    lstm_hidden  : BiLSTM hidden size per direction (output = 2 × lstm_hidden)
    lstm_layers  : number of stacked BiLSTM layers
    lstm_dropout : intra-LSTM and output dropout
    clf_hidden   : shared projection hidden size in TaxonomyClassifier
    proj_dim     : optional input projection dim (None = no projection)
    reg_hidden   : hidden size in PresenceRegressor
    """

    def __init__(
        self,
        input_dim:    int,
        lstm_hidden:  int   = 512,
        lstm_layers:  int   = 2,
        lstm_dropout: float = 0.3,
        clf_hidden:   int   = 256,
        proj_dim:     Optional[int] = None,
        reg_hidden:   int   = 128,
    ) -> None:
        super().__init__()

        self.encoder = BiLSTMContextEncoder(
            input_dim  = input_dim,
            hidden_dim = lstm_hidden,
            num_layers = lstm_layers,
            dropout    = lstm_dropout,
            proj_dim   = proj_dim,
        )
        ctx_dim = self.encoder.output_dim   # 2 × lstm_hidden

        self.classifier1 = TaxonomyClassifier(
            context_dim = ctx_dim,
            hidden_dim  = clf_hidden,
        )

        self.classifier2 = PresenceRegressor(
            context_dim = ctx_dim,
            hidden_dim  = reg_hidden,
        )

    # ── Context embeddings ─────────────────────────────────────────────────

    def get_context(
        self,
        embeddings: torch.Tensor,
        lengths:    List[int],
    ) -> torch.Tensor:
        """Return (B, T, 2H) context-aware embeddings (no classifier heads)."""
        return self.encoder(embeddings, lengths)

    # ── Stage-specific forwards ────────────────────────────────────────────

    def forward_stage1(
        self,
        embeddings: torch.Tensor,
        lengths:    List[int],
    ) -> Dict[str, torch.Tensor]:
        """Full forward for stage 1 training (all params trainable)."""
        ctx = self.encoder(embeddings, lengths)
        return self.classifier1(ctx)

    def forward_stage2(
        self,
        embeddings: torch.Tensor,
        lengths:    List[int],
    ) -> torch.Tensor:
        """
        Forward for stage 2 training.
        Encoder runs under no_grad → its activations are not stored for
        backprop, saving memory.  Classifier2 still receives gradient signal
        through its own weight tensors.
        """
        with torch.no_grad():
            ctx = self.encoder(embeddings, lengths)
        return self.classifier2(ctx)

    def forward(
        self,
        embeddings: torch.Tensor,
        lengths:    List[int],
        stage:      int = 1,
    ):
        if stage == 1:
            return self.forward_stage1(embeddings, lengths)
        return self.forward_stage2(embeddings, lengths)

    # ── Freeze / unfreeze helpers ──────────────────────────────────────────

    def freeze_for_stage2(self) -> None:
        """Freeze encoder + classifier1; leave only classifier2 trainable."""
        for param in self.encoder.parameters():
            param.requires_grad = False
        for param in self.classifier1.parameters():
            param.requires_grad = False
        for param in self.classifier2.parameters():
            param.requires_grad = True
        print("[TimelineModel] Frozen: encoder + classifier1.  Trainable: classifier2.")

    def unfreeze_all(self) -> None:
        for param in self.parameters():
            param.requires_grad = True
        print("[TimelineModel] All parameters unfrozen.")

    def trainable_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def total_params(self) -> int:
        return sum(p.numel() for p in self.parameters())


# ─────────────────────────────────────────────────────────────────────────────
# 8.  Loss functions
# ─────────────────────────────────────────────────────────────────────────────

class Stage1Loss(nn.Module):
    """
    Combined loss for Stage 1:

        L = w_dim  × mean(CrossEntropy over 12 heads)
          + w_se   × (BCE_switch + BCE_escalation)

    All losses are masked to annotated posts only (ann_mask).

    BCE for Switch and Escalation uses positive-class weighting to handle
    the heavy class imbalance (most posts have no event).

    Parameters
    ----------
    dim_ce_weight : weight on the 12 taxonomy heads (default 1.0)
    se_weight     : weight on switch + escalation BCE (default 2.0)
    pos_weight    : BCEWithLogitsLoss positive-class weight for imbalance
    """

    def __init__(
        self,
        dim_ce_weight: float = 1.0,
        se_weight:     float = 2.0,
        pos_weight:    float = 5.0,   # most timelines have few events
    ) -> None:
        super().__init__()
        self.dim_ce_weight = dim_ce_weight
        self.se_weight     = se_weight
        self.register_buffer("pw", torch.tensor([pos_weight]))

    def forward(
        self,
        logits: Dict[str, torch.Tensor],
        batch:  Dict,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        ann_mask  = batch["ann_mask"]    # (B, T) bool
        head_lbl  = batch["head_labels"] # (B, T, 12)
        device    = ann_mask.device

        total   = torch.zeros(1, device=device)
        metrics: Dict[str, float] = {}

        if not ann_mask.any():
            return total.squeeze(), {"total": 0.0}

        flat_ann = ann_mask.reshape(-1)   # (B*T,)

        # ── 12 CrossEntropy heads ────────────────────────────────────────
        ce_sum = torch.zeros(1, device=device)
        for h_idx, (valence, dim) in enumerate(HEAD_ORDER):
            key   = f"{valence}_{dim}"
            logit = logits[key]               # (B, T, n_cls)
            B, T, C = logit.shape
            lf    = logit.reshape(B * T, C)[flat_ann]      # (N_ann, C)
            lblf  = head_lbl[..., h_idx].reshape(B * T)[flat_ann]  # (N_ann,)
            if lblf.numel() == 0:
                continue
            loss = F.cross_entropy(lf, lblf)
            ce_sum   = ce_sum + loss
            metrics[key] = loss.item()

        ce_mean = ce_sum / max(len(HEAD_ORDER), 1)
        total   = total + self.dim_ce_weight * ce_mean

        # ── Switch BCE ───────────────────────────────────────────────────
        sw_logit = logits["switch"].squeeze(-1)   # (B, T)
        sw_lbl   = batch["switch_labels"]          # (B, T)
        sw_loss  = F.binary_cross_entropy_with_logits(
            sw_logit[ann_mask],
            sw_lbl[ann_mask],
            pos_weight=self.pw.to(device),
        )
        total = total + self.se_weight * sw_loss
        metrics["switch"] = sw_loss.item()

        # ── Escalation BCE ───────────────────────────────────────────────
        esc_logit = logits["escalation"].squeeze(-1)
        esc_lbl   = batch["escalation_labels"]
        esc_loss  = F.binary_cross_entropy_with_logits(
            esc_logit[ann_mask],
            esc_lbl[ann_mask],
            pos_weight=self.pw.to(device),
        )
        total = total + self.se_weight * esc_loss
        metrics["escalation"] = esc_loss.item()

        metrics["ce_mean"]    = ce_mean.item()
        metrics["total"]      = total.item()
        return total.squeeze(), metrics


class Stage2Loss(nn.Module):
    """
    RMSE loss for adaptive and maladaptive presence scores,
    masked to annotated posts only.

    RMSE is preferred over plain MSE here because the target scale (1-5)
    is small and we want the loss magnitude to be directly interpretable
    in 'presence units'.
    """

    def forward(
        self,
        preds: torch.Tensor,   # (B, T, 2)
        batch: Dict,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        ann_mask = batch["ann_mask"]   # (B, T)
        device   = preds.device

        if not ann_mask.any():
            z = torch.zeros(1, device=device)
            return z.squeeze(), {"ada_rmse": 0.0, "mal_rmse": 0.0, "total": 0.0}

        ada_pred = preds[ann_mask, 0]            # (N_ann,)
        mal_pred = preds[ann_mask, 1]            # (N_ann,)
        ada_tgt  = batch["ada_presence"][ann_mask]
        mal_tgt  = batch["mal_presence"][ann_mask]

        ada_rmse = torch.sqrt(F.mse_loss(ada_pred, ada_tgt) + 1e-8)
        mal_rmse = torch.sqrt(F.mse_loss(mal_pred, mal_tgt) + 1e-8)
        total    = (ada_rmse + mal_rmse) / 2.0

        return total, {
            "ada_rmse": ada_rmse.item(),
            "mal_rmse": mal_rmse.item(),
            "total":    total.item(),
        }


# ─────────────────────────────────────────────────────────────────────────────
# 9.  Trainer
# ─────────────────────────────────────────────────────────────────────────────

class TimelineModelTrainer:
    """
    Two-stage trainer.

    Stage 1 trains the encoder + TaxonomyClassifier end-to-end with
    a combined CrossEntropy + BCE objective.

    Stage 2 freezes the encoder + classifier1 and trains only the
    PresenceRegressor with RMSE loss.  The encoder runs under no_grad
    during stage 2, so memory usage stays low.

    Parameters
    ----------
    model           : TimelineModel
    device          : "cuda" | "cpu" | "mps"
    lr_stage1       : AdamW learning rate for stage 1
    lr_stage2       : AdamW learning rate for stage 2
    weight_decay    : L2 regularisation
    dim_ce_weight   : weight on taxonomy CrossEntropy in stage 1 loss
    se_weight       : weight on Switch/Escalation BCE in stage 1 loss
    pos_weight      : BCEWithLogitsLoss positive-class weight for events
    """

    def __init__(
        self,
        model:          TimelineModel,
        device:         str   = "cpu",
        lr_stage1:      float = 1e-3,
        lr_stage2:      float = 5e-4,
        weight_decay:   float = 1e-4,
        dim_ce_weight:  float = 1.0,
        se_weight:      float = 2.0,
        pos_weight:     float = 5.0,
    ) -> None:
        self.model    = model.to(device)
        self.device   = device
        self.lr_s1    = lr_stage1
        self.lr_s2    = lr_stage2
        self.wd       = weight_decay
        self._loss1   = Stage1Loss(dim_ce_weight, se_weight, pos_weight)
        self._loss2   = Stage2Loss()

    # ── Stage 1 ───────────────────────────────────────────────────────────

    def train_stage1(
        self,
        train_loader: DataLoader,
        val_loader:   Optional[DataLoader] = None,
        epochs:       int  = 15,
        save_path:    str  = "timeline_model_s1.pt",
    ) -> None:
        """Train encoder + classifier1 end-to-end."""
        self.model.unfreeze_all()
        print(f"[Stage 1] Trainable params: {self.model.trainable_params():,}")

        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr_s1, weight_decay=self.wd)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=self.lr_s1,
            steps_per_epoch=len(train_loader),
            epochs=epochs,
            pct_start=0.1,
            anneal_strategy="cos",
        )
        self._fit(train_loader, val_loader, epochs, save_path, opt, scheduler, stage=1)

    # ── Stage 2 ───────────────────────────────────────────────────────────

    def train_stage2(
        self,
        train_loader: DataLoader,
        val_loader:   Optional[DataLoader] = None,
        epochs:       int  = 10,
        save_path:    str  = "timeline_model_s2.pt",
    ) -> None:
        """Freeze encoder + classifier1; train only PresenceRegressor."""
        self.model.freeze_for_stage2()
        print(f"[Stage 2] Trainable params: {self.model.trainable_params():,}")

        opt = torch.optim.AdamW(
            self.model.classifier2.parameters(), lr=self.lr_s2, weight_decay=self.wd
        )
        self._fit(train_loader, val_loader, epochs, save_path, opt, scheduler=None, stage=2)

    # ── Internal loop ────────────────────────────────────────────────────

    def _fit(
        self,
        train_loader: DataLoader,
        val_loader:   Optional[DataLoader],
        epochs:       int,
        save_path:    str,
        opt:          torch.optim.Optimizer,
        scheduler,
        stage:        int,
    ) -> None:
        best_val = math.inf
        for epoch in range(1, epochs + 1):
            tr  = self._epoch(train_loader, opt, scheduler, stage, train=True)
            log = f"[S{stage}] Epoch {epoch:03d}/{epochs} | train {self._fmt(tr)}"

            if val_loader is not None:
                vl  = self._epoch(val_loader, stage=stage, train=False)
                log += f" || val {self._fmt(vl)}"
                monitor = vl["total"]
            else:
                monitor = tr["total"]

            if monitor < best_val:
                best_val = monitor
                torch.save(self.model.state_dict(), save_path)
                log += "  ✓ saved"

            print(log)

    def _epoch(
        self,
        loader:    DataLoader,
        opt        = None,
        scheduler  = None,
        stage:     int  = 1,
        train:     bool = True,
    ) -> Dict[str, float]:
        self.model.train(train)
        totals: Dict[str, float] = {}

        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for batch in loader:
                batch = {
                    k: v.to(self.device) if isinstance(v, torch.Tensor) else v
                    for k, v in batch.items()
                }
                emb     = batch["embeddings"]
                lengths = batch["lengths"]

                if stage == 1:
                    logits = self.model.forward_stage1(emb, lengths)
                    loss, metrics = self._loss1(logits, batch)
                else:
                    preds  = self.model.forward_stage2(emb, lengths)
                    loss, metrics = self._loss2(preds, batch)

                if train and opt is not None and loss.requires_grad:
                    opt.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                    opt.step()
                    if scheduler is not None:
                        scheduler.step()

                for k, v in metrics.items():
                    totals[k] = totals.get(k, 0.0) + v

        n = max(len(loader), 1)
        return {k: v / n for k, v in totals.items()}

    @staticmethod
    def _fmt(m: Dict[str, float]) -> str:
        important = ["total", "ce_mean", "switch", "escalation", "ada_rmse", "mal_rmse"]
        parts     = [f"{k}={m[k]:.4f}" for k in important if k in m]
        return "  ".join(parts)


# ─────────────────────────────────────────────────────────────────────────────
# 10. Inference utilities
# ─────────────────────────────────────────────────────────────────────────────

# Inverse maps: local_idx → subelement number (or None)
_INV_HEAD: Dict[str, Dict[str, Dict[int, Optional[int]]]] = {
    valence: {
        dim: {local_idx: number for number, local_idx in idx_map.items()}
        for dim, idx_map in dim_map.items()
    }
    for valence, dim_map in HEAD_LABEL_MAP.items()
}


class TimelineInference:
    """
    Wraps a trained TimelineModel for easy inference.

    Methods
    -------
    get_context_embeddings(embeddings) → (T, 2H) numpy array
        Context-aware per-post embeddings — append to PostEmbedder output.

    predict(embeddings, threshold) → List[Dict]
        Full decoded predictions for every post in the timeline.
    """

    def __init__(self, model: TimelineModel, device: str = "cpu") -> None:
        self.model  = model.to(device).eval()
        self.device = device

    @torch.no_grad()
    def get_context_embeddings(self, embeddings: np.ndarray) -> np.ndarray:
        """
        Parameters
        ----------
        embeddings : (T, D) float32 – post embeddings for one timeline

        Returns
        -------
        (T, 2H) float32 numpy array — one context-aware vector per post.
        Concatenate with post.post_embedding and/or SelfStateEmbedder output
        before feeding to TopKSimilarDataset or downstream classifiers.
        """
        t = torch.tensor(embeddings, dtype=torch.float32).unsqueeze(0).to(self.device)
        ctx = self.model.encoder(t, [embeddings.shape[0]])  # (1, T, 2H)
        return ctx.squeeze(0).cpu().numpy()

    @torch.no_grad()
    def predict(
        self,
        embeddings: np.ndarray,
        threshold:  float = 0.5,
    ) -> List[Dict]:
        """
        Parameters
        ----------
        embeddings : (T, D) float32 – post embeddings for one timeline
        threshold  : sigmoid threshold for Switch and Escalation (default 0.5)

        Returns
        -------
        List of T dicts, one per post:
        {
          "adaptive":     {"A": int|None, "B-O": int|None, ...},
          "maladaptive":  {"A": int|None, "B-O": int|None, ...},
          "switch":       bool,
          "escalation":   bool,
          "ada_presence": float,   # in [1, 5]
          "mal_presence": float,   # in [1, 5]
        }
        None in adaptive/maladaptive dicts means "not present in this dimension".
        """
        t       = torch.tensor(embeddings, dtype=torch.float32).unsqueeze(0).to(self.device)
        lengths = [embeddings.shape[0]]

        ctx    = self.model.encoder(t, lengths)       # (1, T, 2H)
        logits = self.model.classifier1(ctx)
        pres   = self.model.classifier2(ctx)          # (1, T, 2)

        T = embeddings.shape[0]
        results = []

        for t_idx in range(T):
            pred: Dict = {"adaptive": {}, "maladaptive": {}}

            for valence, dim in HEAD_ORDER:
                key      = f"{valence}_{dim}"
                logit_t  = logits[key][0, t_idx]             # (n_cls,)
                local_idx = logit_t.argmax().item()
                pred[valence][dim] = _INV_HEAD[valence][dim][local_idx]

            pred["switch"]      = logits["switch"][0, t_idx, 0].sigmoid().item() >= threshold
            pred["escalation"]  = logits["escalation"][0, t_idx, 0].sigmoid().item() >= threshold
            pred["ada_presence"] = float(pres[0, t_idx, 0].item())
            pred["mal_presence"] = float(pres[0, t_idx, 1].item())

            results.append(pred)

        return results

    @torch.no_grad()
    def enrich_timeline(
        self,
        timeline_embeddings: np.ndarray,      # (T, D)
    ) -> np.ndarray:                          # (T, D + 2H)
        """
        Append context embeddings to the original post embeddings.
        Use this as a drop-in replacement for timeline post embeddings
        before building PostIndex / TopKSimilarDataset.
        """
        ctx = self.get_context_embeddings(timeline_embeddings)
        return np.concatenate([timeline_embeddings, ctx], axis=1)


# ─────────────────────────────────────────────────────────────────────────────
# 11. Quick-start  (python timeline_model.py)
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import random
    from torch.utils.data import random_split
    from data_structure import load_all_timelines

    # ── Config ────────────────────────────────────────────────────────────
    DATA_DIR     = "../../../data/train/"
    DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
    SEED         = 42
    BATCH_SIZE   = 8        # one sample = one full timeline

    # Model hyper-parameters
    # input_dim must match post_embedding dimensionality produced by PostEmbedder
    # (300 Word2Vec + 1024 sentence-transformer + 310 task scores = 1634 by default)
    INPUT_DIM    = 1357
    LSTM_HIDDEN  = 512      # output context dim = 2 × 512 = 1024
    LSTM_LAYERS  = 2
    PROJ_DIM     = 512      # project input before LSTM (reduces params)

    random.seed(SEED)
    torch.manual_seed(SEED)

    # ── Data ──────────────────────────────────────────────────────────────
    timelines = load_all_timelines(DATA_DIR)

    full_ds = TimelineDataset(timelines, skip_no_embedding=True)

    n_train = int(0.8 * len(full_ds))
    n_val   = len(full_ds) - n_train
    train_ds, val_ds = random_split(
        full_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED),
    )

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=TimelineDataset.collate,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=TimelineDataset.collate,
    )

    # ── Model ─────────────────────────────────────────────────────────────
    model = TimelineModel(
        input_dim    = INPUT_DIM,
        lstm_hidden  = LSTM_HIDDEN,
        lstm_layers  = LSTM_LAYERS,
        lstm_dropout = 0.3,
        clf_hidden   = 256,
        proj_dim     = PROJ_DIM,
        reg_hidden   = 128,
    )
    print(f"[Model] Total params: {model.total_params():,}")

    # ── Stage 1 ───────────────────────────────────────────────────────────
    trainer = TimelineModelTrainer(
        model         = model,
        device        = DEVICE,
        lr_stage1     = 1e-3,
        lr_stage2     = 5e-4,
        dim_ce_weight = 1.0,
        se_weight     = 2.0,
        pos_weight    = 5.0,
    )
    trainer.train_stage1(train_loader, val_loader, epochs=50, save_path="timeline_model_s1.pt")

    # ── Stage 2 ───────────────────────────────────────────────────────────
    # Load best stage-1 checkpoint before freezing
    model.load_state_dict(torch.load("timeline_model_s1.pt", map_location=DEVICE))
    trainer.train_stage2(train_loader, val_loader, epochs=50, save_path="timeline_model_s2.pt")

    # ── Inference demo ────────────────────────────────────────────────────
    model.load_state_dict(torch.load("timeline_model_s2.pt", map_location=DEVICE))
    infer = TimelineInference(model, device=DEVICE)

    tl = timelines[0]
    emb_matrix = np.stack([p.post_embedding for p in tl.posts if p.post_embedding is not None])
    predictions = infer.predict(emb_matrix)

    print(f"\nTimeline {tl.timeline_id!r}: {len(predictions)} post predictions")
    for i, pred in enumerate(predictions[:3]):
        print(f"  Post {i}: switch={pred['switch']}  esc={pred['escalation']}"
              f"  ada_pres={pred['ada_presence']:.2f}  mal_pres={pred['mal_presence']:.2f}")
        print(f"    adaptive:    {pred['adaptive']}")
        print(f"    maladaptive: {pred['maladaptive']}")

    # ── Enrich embeddings for downstream use ──────────────────────────────
    enriched = infer.enrich_timeline(emb_matrix)
    print(f"\nOriginal embedding dim : {emb_matrix.shape[1]}")
    print(f"Enriched embedding dim : {enriched.shape[1]}  (+ {2*LSTM_HIDDEN} context dims)")


Loaded 30 timelines from: ../../../data/train/
  Timelines             : 30
  Total posts           : 373
  Annotated posts       : 236
  Switch posts          : 78
  Escalation posts      : 75
  Both (S+E) posts      : 27
  Adaptive subelements  : 472
  Maladaptive subelements: 615
[TimelineDataset] 30 timelines loaded  (0 skipped).
[Model] Total params: 11,615,088
[TimelineModel] All parameters unfrozen.
[Stage 1] Trainable params: 11,615,088
[S1] Epoch 001/50 | train total=6.9881  ce_mean=1.1996  switch=1.4359  escalation=1.4583 || val total=6.7472  ce_mean=1.1377  switch=1.3637  escalation=1.4410  ✓ saved
[S1] Epoch 002/50 | train total=6.2299  ce_mean=1.1099  switch=1.3389  escalation=1.2211 || val total=8.1898  ce_mean=1.0250  switch=1.5819  escalation=2.0005
[S1] Epoch 003/50 | train total=6.1683  ce_mean=1.0113  switch=1.3829  escalation=1.1956 || val total=6.9432  ce_mean=0.9551  switch=1.4068  escalation=1.5873
[S1] Epoch 004/50 | train total=5.5014  ce_mean=0.9446  switch=1

In [7]:
# timeline_model.py
"""
BiLSTM Timeline Context Model
==============================
Processes an entire timeline of post embeddings through a BiLSTM to produce
context-aware embeddings, then passes them through two staged classifiers.

Architecture
------------
  post_embeddings (T, D)
         ↓
  [Optional Input Projection]  D → proj_dim
         ↓
  BiLSTM (num_layers, bidirectional)
         ↓
  context_embeddings (T, 2H)   ← one per post, full temporal context
         ↓
  ┌──────────────────────────────────────────────────────────────────────┐
  │  STAGE 1 — TaxonomyClassifier                                        │
  │  12 × CrossEntropy heads  (one per valence × dimension)              │
  │    adaptive  : A(8), B-O(3), B-S(2), C-O(3), C-S(2), D(4)          │
  │    maladaptive: A(8), B-O(3), B-S(2), C-O(3), C-S(2), D(4)         │
  │  2 × BCE heads  (Switch, Escalation)  — threshold 0.5               │
  └──────────────────────────────────────────────────────────────────────┘
         ↓  (freeze encoder + classifier1)
  ┌──────────────────────────────────────────────────────────────────────┐
  │  STAGE 2 — PresenceRegressor  (RMSE loss)                           │
  │    adaptive_presence  (1-5)                                          │
  │    maladaptive_presence (1-5)                                        │
  └──────────────────────────────────────────────────────────────────────┘
"""

from __future__ import annotations

import math
import os
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

from data_structure import Post, Timeline

# ─────────────────────────────────────────────────────────────────────────────
# 1.  Label definitions
# ─────────────────────────────────────────────────────────────────────────────

DIMENSIONS = ["A", "B-O", "B-S", "C-O", "C-S", "D"]

# Per-head LOCAL class indices (None = "not present in this dimension")
# These are the targets passed to CrossEntropyLoss for each head.
HEAD_LABEL_MAP: Dict[str, Dict[str, Dict]] = {
    "adaptive": {
        "A":   {1: 0, 3: 1, 5: 2,  7: 3,  9: 4,  11: 5, 13: 6, None: 7},
        "B-O": {1: 0, 3: 1, None: 2},
        "B-S": {1: 0, None: 1},
        "C-O": {1: 0, 3: 1, None: 2},
        "C-S": {1: 0, None: 1},
        "D":   {1: 0, 3: 1, 5: 2,  None: 3},
    },
    "maladaptive": {
        "A":   {2: 0, 4: 1, 6: 2,  8: 3,  10: 4, 12: 5, 14: 6, None: 7},
        "B-O": {2: 0, 4: 1, None: 2},
        "B-S": {2: 0, None: 1},
        "C-O": {2: 0, 4: 1, None: 2},
        "C-S": {2: 0, None: 1},
        "D":   {2: 0, 4: 1, 6: 2,  None: 3},
    },
}

# Number of output classes per head
HEAD_SIZES: Dict[str, Dict[str, int]] = {
    valence: {dim: max(idx_map.values()) + 1 for dim, idx_map in dim_map.items()}
    for valence, dim_map in HEAD_LABEL_MAP.items()
}

# Canonical ordered list of (valence, dim) pairs → 12 heads total
HEAD_ORDER: List[Tuple[str, str]] = [
    (v, d) for v in ["adaptive", "maladaptive"] for d in DIMENSIONS
]

PRESENCE_MIN = 1.0
PRESENCE_MAX = 5.0


# ─────────────────────────────────────────────────────────────────────────────
# 2.  Label conversion helpers
# ─────────────────────────────────────────────────────────────────────────────

def _post_to_head_labels(post: Post) -> np.ndarray:
    """
    Returns (12,) int64 array of per-head local class indices.
    """
    labels = np.empty(12, dtype=np.int64)
    for h_idx, (valence, dim) in enumerate(HEAD_ORDER):
        state   = post.adaptive_state if valence == "adaptive" else post.maladaptive_state
        idx_map = HEAD_LABEL_MAP[valence][dim]
        none_idx = idx_map[None]
        se_map  = state.by_dimension   # {dim_str: SubElement}
        if dim in se_map:
            labels[h_idx] = idx_map.get(se_map[dim].number, none_idx)
        else:
            labels[h_idx] = none_idx
    return labels


# ─────────────────────────────────────────────────────────────────────────────
# 3.  Dataset
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class TimelineSample:
    timeline_id:       str
    embeddings:        np.ndarray   # (T, D)  float32
    head_labels:       np.ndarray   # (T, 12) int64   
    switch_labels:     np.ndarray   # (T,)    float32 
    escalation_labels: np.ndarray   # (T,)    float32 
    ada_presence:      np.ndarray   # (T,)    float32 
    mal_presence:      np.ndarray   # (T,)    float32 
    is_annotated:      np.ndarray   # (T,)    bool    


class TimelineDataset(Dataset):
    def __init__(
        self,
        timelines:         List[Timeline],
        skip_no_embedding: bool = True,
    ) -> None:
        self._samples: List[TimelineSample] = []
        skipped = 0

        for tl in timelines:
            posts = tl.posts
            if not posts:
                skipped += 1
                continue
            if skip_no_embedding and any(p.post_embedding is None for p in posts):
                skipped += 1
                continue

            embeddings        = np.stack([p.post_embedding for p in posts], axis=0).astype(np.float32)
            head_labels       = np.stack([_post_to_head_labels(p) for p in posts], axis=0)
            switch_labels     = np.array([1.0 if p.is_switch     else 0.0 for p in posts], dtype=np.float32)
            escalation_labels = np.array([1.0 if p.is_escalation else 0.0 for p in posts], dtype=np.float32)
            ada_presence      = np.array([float(p.adaptive_state.presence)    for p in posts], dtype=np.float32)
            mal_presence      = np.array([float(p.maladaptive_state.presence) for p in posts], dtype=np.float32)
            is_annotated      = np.array([p.is_annotated for p in posts], dtype=bool)

            self._samples.append(TimelineSample(
                timeline_id       = tl.timeline_id,
                embeddings        = embeddings,
                head_labels       = head_labels,
                switch_labels     = switch_labels,
                escalation_labels = escalation_labels,
                ada_presence      = ada_presence,
                mal_presence      = mal_presence,
                is_annotated      = is_annotated,
            ))

        print(f"[TimelineDataset] {len(self._samples)} timelines loaded  ({skipped} skipped).")

    def __len__(self) -> int:
        return len(self._samples)

    def __getitem__(self, idx: int) -> TimelineSample:
        return self._samples[idx]

    @staticmethod
    def collate(batch: List[TimelineSample]) -> Dict:
        lengths = [s.embeddings.shape[0] for s in batch]
        T_max   = max(lengths)
        D       = batch[0].embeddings.shape[1]
        B       = len(batch)

        emb_pad  = torch.zeros(B, T_max, D,   dtype=torch.float32)
        hl_pad   = torch.zeros(B, T_max, 12,  dtype=torch.long)
        sw_pad   = torch.zeros(B, T_max,      dtype=torch.float32)
        esc_pad  = torch.zeros(B, T_max,      dtype=torch.float32)
        ada_pad  = torch.ones(B,  T_max,      dtype=torch.float32)
        mal_pad  = torch.ones(B,  T_max,      dtype=torch.float32)
        ann_mask = torch.zeros(B, T_max,      dtype=torch.bool)
        pad_mask = torch.zeros(B, T_max,      dtype=torch.bool)

        for i, s in enumerate(batch):
            T = s.embeddings.shape[0]
            emb_pad[i, :T]  = torch.from_numpy(s.embeddings)
            hl_pad[i, :T]   = torch.from_numpy(s.head_labels)
            sw_pad[i, :T]   = torch.from_numpy(s.switch_labels)
            esc_pad[i, :T]  = torch.from_numpy(s.escalation_labels)
            ada_pad[i, :T]  = torch.from_numpy(s.ada_presence)
            mal_pad[i, :T]  = torch.from_numpy(s.mal_presence)
            ann_mask[i, :T] = torch.from_numpy(s.is_annotated)
            pad_mask[i, :T] = True

        return {
            "embeddings":        emb_pad,
            "lengths":           lengths,
            "head_labels":       hl_pad,
            "switch_labels":     sw_pad,
            "escalation_labels": esc_pad,
            "ada_presence":      ada_pad,
            "mal_presence":      mal_pad,
            "ann_mask":          ann_mask,
            "pad_mask":          pad_mask,
        }


# ─────────────────────────────────────────────────────────────────────────────
# 4.  BiLSTM Context Encoder
# ─────────────────────────────────────────────────────────────────────────────

class BiLSTMContextEncoder(nn.Module):
    def __init__(
        self,
        input_dim:  int,
        hidden_dim: int   = 512,
        num_layers: int   = 2,
        dropout:    float = 0.3,
        proj_dim:   Optional[int] = None,
    ) -> None:
        super().__init__()

        if proj_dim is not None and proj_dim != input_dim:
            self.input_proj: nn.Module = nn.Sequential(
                nn.Linear(input_dim, proj_dim),
                nn.LayerNorm(proj_dim),
                nn.GELU(),
            )
            lstm_in = proj_dim
        else:
            self.input_proj = nn.Identity()
            lstm_in = input_dim

        self.lstm = nn.LSTM(
            input_size    = lstm_in,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout if num_layers > 1 else 0.0,
        )

        self.out_norm  = nn.LayerNorm(2 * hidden_dim)
        self.drop      = nn.Dropout(dropout)
        self.output_dim = 2 * hidden_dim

    def forward(
        self,
        embeddings: torch.Tensor,
        lengths:    List[int],
    ) -> torch.Tensor:
        x = self.input_proj(embeddings)                                   
        packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out_packed, _ = self.lstm(packed)
        out, _        = pad_packed_sequence(out_packed, batch_first=True) 
        out = self.out_norm(out)
        out = self.drop(out)
        return out                                                         


# ─────────────────────────────────────────────────────────────────────────────
# 5.  Classifier 1 — Taxonomy + Switch/Escalation
# ─────────────────────────────────────────────────────────────────────────────

class TaxonomyClassifier(nn.Module):
    def __init__(
        self,
        context_dim: int,
        hidden_dim:  int   = 256,
        dropout:     float = 0.2,
    ) -> None:
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(context_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.dim_heads = nn.ModuleDict({
            f"{valence}_{dim}": nn.Linear(hidden_dim, HEAD_SIZES[valence][dim])
            for valence, dim in HEAD_ORDER
        })

        self.switch_head     = nn.Linear(hidden_dim, 1)
        self.escalation_head = nn.Linear(hidden_dim, 1)

    def forward(self, context: torch.Tensor) -> Dict[str, torch.Tensor]:
        h = self.shared(context)   

        logits: Dict[str, torch.Tensor] = {}
        for valence, dim in HEAD_ORDER:
            key = f"{valence}_{dim}"
            logits[key] = self.dim_heads[key](h)   

        logits["switch"]     = self.switch_head(h)       
        logits["escalation"] = self.escalation_head(h)   

        return logits


# ─────────────────────────────────────────────────────────────────────────────
# 6.  Classifier 2 — Presence Regressor  (Stage 2, frozen backbone)
# ─────────────────────────────────────────────────────────────────────────────

class PresenceRegressor(nn.Module):
    def __init__(
        self,
        context_dim: int,
        hidden_dim:  int   = 128,
        dropout:     float = 0.1,
    ) -> None:
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(context_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 2),
        )

    def forward(self, context: torch.Tensor) -> torch.Tensor:
        """
        context : (B, T, context_dim)   (may be a no_grad tensor)
        returns : (B, T, 2)  bounded smoothly to [1, 5]
        """
        raw = self.net(context)                       # (B, T, 2)
        
        # FIXED: Replaced torch.clamp with a scaled Sigmoid to keep gradients alive.
        # Sigmoid bounds to [0, 1]. Multiply by 4 and add 1 to get [1.0, 5.0].
        return 4.0 * torch.sigmoid(raw) + 1.0


# ─────────────────────────────────────────────────────────────────────────────
# 7.  Full Model
# ─────────────────────────────────────────────────────────────────────────────

class TimelineModel(nn.Module):
    def __init__(
        self,
        input_dim:    int,
        lstm_hidden:  int   = 512,
        lstm_layers:  int   = 2,
        lstm_dropout: float = 0.3,
        clf_hidden:   int   = 256,
        proj_dim:     Optional[int] = None,
        reg_hidden:   int   = 128,
    ) -> None:
        super().__init__()

        self.encoder = BiLSTMContextEncoder(
            input_dim  = input_dim,
            hidden_dim = lstm_hidden,
            num_layers = lstm_layers,
            dropout    = lstm_dropout,
            proj_dim   = proj_dim,
        )
        ctx_dim = self.encoder.output_dim   

        self.classifier1 = TaxonomyClassifier(
            context_dim = ctx_dim,
            hidden_dim  = clf_hidden,
        )

        self.classifier2 = PresenceRegressor(
            context_dim = ctx_dim,
            hidden_dim  = reg_hidden,
        )

    def get_context(
        self,
        embeddings: torch.Tensor,
        lengths:    List[int],
    ) -> torch.Tensor:
        return self.encoder(embeddings, lengths)

    def forward_stage1(
        self,
        embeddings: torch.Tensor,
        lengths:    List[int],
    ) -> Dict[str, torch.Tensor]:
        ctx = self.encoder(embeddings, lengths)
        return self.classifier1(ctx)

    def forward_stage2(
        self,
        embeddings: torch.Tensor,
        lengths:    List[int],
    ) -> torch.Tensor:
        with torch.no_grad():
            ctx = self.encoder(embeddings, lengths)
        return self.classifier2(ctx)

    def forward(
        self,
        embeddings: torch.Tensor,
        lengths:    List[int],
        stage:      int = 1,
    ):
        if stage == 1:
            return self.forward_stage1(embeddings, lengths)
        return self.forward_stage2(embeddings, lengths)

    def freeze_for_stage2(self) -> None:
        for param in self.encoder.parameters():
            param.requires_grad = False
        for param in self.classifier1.parameters():
            param.requires_grad = False
        for param in self.classifier2.parameters():
            param.requires_grad = True
        print("[TimelineModel] Frozen: encoder + classifier1.  Trainable: classifier2.")

    def unfreeze_all(self) -> None:
        for param in self.parameters():
            param.requires_grad = True
        print("[TimelineModel] All parameters unfrozen.")

    def trainable_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def total_params(self) -> int:
        return sum(p.numel() for p in self.parameters())


# ─────────────────────────────────────────────────────────────────────────────
# 8.  Loss functions
# ─────────────────────────────────────────────────────────────────────────────

class Stage1Loss(nn.Module):
    def __init__(
        self,
        dim_ce_weight: float = 1.0,
        se_weight:     float = 2.0,
        pos_weight:    float = 5.0,   
    ) -> None:
        super().__init__()
        self.dim_ce_weight = dim_ce_weight
        self.se_weight     = se_weight
        self.register_buffer("pw", torch.tensor([pos_weight]))

    def forward(
        self,
        logits: Dict[str, torch.Tensor],
        batch:  Dict,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        ann_mask  = batch["ann_mask"]    
        head_lbl  = batch["head_labels"] 
        device    = ann_mask.device

        total   = torch.zeros(1, device=device)
        metrics: Dict[str, float] = {}

        if not ann_mask.any():
            return total.squeeze(), {"total": 0.0}

        flat_ann = ann_mask.reshape(-1)   

        ce_sum = torch.zeros(1, device=device)
        for h_idx, (valence, dim) in enumerate(HEAD_ORDER):
            key   = f"{valence}_{dim}"
            logit = logits[key]               
            B, T, C = logit.shape
            lf    = logit.reshape(B * T, C)[flat_ann]      
            lblf  = head_lbl[..., h_idx].reshape(B * T)[flat_ann]  
            if lblf.numel() == 0:
                continue
            loss = F.cross_entropy(lf, lblf)
            ce_sum   = ce_sum + loss
            metrics[key] = loss.item()

        ce_mean = ce_sum / max(len(HEAD_ORDER), 1)
        total   = total + self.dim_ce_weight * ce_mean

        sw_logit = logits["switch"].squeeze(-1)   
        sw_lbl   = batch["switch_labels"]          
        sw_loss  = F.binary_cross_entropy_with_logits(
            sw_logit[ann_mask],
            sw_lbl[ann_mask],
            pos_weight=self.pw.to(device),
        )
        total = total + self.se_weight * sw_loss
        metrics["switch"] = sw_loss.item()

        esc_logit = logits["escalation"].squeeze(-1)
        esc_lbl   = batch["escalation_labels"]
        esc_loss  = F.binary_cross_entropy_with_logits(
            esc_logit[ann_mask],
            esc_lbl[ann_mask],
            pos_weight=self.pw.to(device),
        )
        total = total + self.se_weight * esc_loss
        metrics["escalation"] = esc_loss.item()

        metrics["ce_mean"]    = ce_mean.item()
        metrics["total"]      = total.item()
        return total.squeeze(), metrics


class Stage2Loss(nn.Module):
    def forward(
        self,
        preds: torch.Tensor,   # (B, T, 2)
        batch: Dict,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        ann_mask = batch["ann_mask"]   # (B, T)
        device   = preds.device

        if not ann_mask.any():
            z = torch.zeros(1, device=device)
            return z.squeeze(), {"ada_rmse": 0.0, "mal_rmse": 0.0, "total": 0.0}

        ada_pred = preds[ann_mask, 0]            
        mal_pred = preds[ann_mask, 1]            
        ada_tgt  = batch["ada_presence"][ann_mask]
        mal_tgt  = batch["mal_presence"][ann_mask]

        ada_rmse = torch.sqrt(F.mse_loss(ada_pred, ada_tgt) + 1e-8)
        mal_rmse = torch.sqrt(F.mse_loss(mal_pred, mal_tgt) + 1e-8)
        total    = (ada_rmse + mal_rmse) / 2.0

        return total, {
            "ada_rmse": ada_rmse.item(),
            "mal_rmse": mal_rmse.item(),
            "total":    total.item(),
        }


# ─────────────────────────────────────────────────────────────────────────────
# 9.  Trainer
# ─────────────────────────────────────────────────────────────────────────────

class TimelineModelTrainer:
    def __init__(
        self,
        model:          TimelineModel,
        device:         str   = "cpu",
        lr_stage1:      float = 1e-3,
        lr_stage2:      float = 5e-4,
        weight_decay:   float = 1e-4,
        dim_ce_weight:  float = 1.0,
        se_weight:      float = 2.0,
        pos_weight:     float = 5.0,
    ) -> None:
        self.model    = model.to(device)
        self.device   = device
        self.lr_s1    = lr_stage1
        self.lr_s2    = lr_stage2
        self.wd       = weight_decay
        self._loss1   = Stage1Loss(dim_ce_weight, se_weight, pos_weight)
        self._loss2   = Stage2Loss()

    def train_stage1(
        self,
        train_loader: DataLoader,
        val_loader:   Optional[DataLoader] = None,
        epochs:       int  = 15,
        save_path:    str  = "timeline_model_s1.pt",
    ) -> None:
        self.model.unfreeze_all()
        print(f"[Stage 1] Trainable params: {self.model.trainable_params():,}")

        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr_s1, weight_decay=self.wd)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=self.lr_s1,
            steps_per_epoch=len(train_loader),
            epochs=epochs,
            pct_start=0.1,
            anneal_strategy="cos",
        )
        self._fit(train_loader, val_loader, epochs, save_path, opt, scheduler, stage=1)

    def train_stage2(
        self,
        train_loader: DataLoader,
        val_loader:   Optional[DataLoader] = None,
        epochs:       int  = 10,
        save_path:    str  = "timeline_model_s2.pt",
    ) -> None:
        self.model.freeze_for_stage2()
        print(f"[Stage 2] Trainable params: {self.model.trainable_params():,}")

        opt = torch.optim.AdamW(
            self.model.classifier2.parameters(), lr=self.lr_s2, weight_decay=self.wd
        )
        self._fit(train_loader, val_loader, epochs, save_path, opt, scheduler=None, stage=2)

    def _fit(
        self,
        train_loader: DataLoader,
        val_loader:   Optional[DataLoader],
        epochs:       int,
        save_path:    str,
        opt:          torch.optim.Optimizer,
        scheduler,
        stage:        int,
    ) -> None:
        best_val = math.inf
        for epoch in range(1, epochs + 1):
            tr  = self._epoch(train_loader, opt, scheduler, stage, train=True)
            log = f"[S{stage}] Epoch {epoch:03d}/{epochs} | train {self._fmt(tr)}"

            if val_loader is not None:
                vl  = self._epoch(val_loader, stage=stage, train=False)
                log += f" || val {self._fmt(vl)}"
                monitor = vl["total"]
            else:
                monitor = tr["total"]

            if monitor < best_val:
                best_val = monitor
                torch.save(self.model.state_dict(), save_path)
                log += "  ✓ saved"

            print(log)

    def _epoch(
        self,
        loader:    DataLoader,
        opt        = None,
        scheduler  = None,
        stage:     int  = 1,
        train:     bool = True,
    ) -> Dict[str, float]:
        self.model.train(train)
        totals: Dict[str, float] = {}

        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for batch in loader:
                batch = {
                    k: v.to(self.device) if isinstance(v, torch.Tensor) else v
                    for k, v in batch.items()
                }
                emb     = batch["embeddings"]
                lengths = batch["lengths"]

                if stage == 1:
                    logits = self.model.forward_stage1(emb, lengths)
                    loss, metrics = self._loss1(logits, batch)
                else:
                    preds  = self.model.forward_stage2(emb, lengths)
                    loss, metrics = self._loss2(preds, batch)

                if train and opt is not None and loss.requires_grad:
                    opt.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                    opt.step()
                    if scheduler is not None:
                        scheduler.step()

                for k, v in metrics.items():
                    totals[k] = totals.get(k, 0.0) + v

        n = max(len(loader), 1)
        return {k: v / n for k, v in totals.items()}

    @staticmethod
    def _fmt(m: Dict[str, float]) -> str:
        important = ["total", "ce_mean", "switch", "escalation", "ada_rmse", "mal_rmse"]
        parts     = [f"{k}={m[k]:.4f}" for k in important if k in m]
        return "  ".join(parts)


# ─────────────────────────────────────────────────────────────────────────────
# 10. Inference utilities
# ─────────────────────────────────────────────────────────────────────────────

_INV_HEAD: Dict[str, Dict[str, Dict[int, Optional[int]]]] = {
    valence: {
        dim: {local_idx: number for number, local_idx in idx_map.items()}
        for dim, idx_map in dim_map.items()
    }
    for valence, dim_map in HEAD_LABEL_MAP.items()
}


class TimelineInference:
    def __init__(self, model: TimelineModel, device: str = "cpu") -> None:
        self.model  = model.to(device).eval()
        self.device = device

    @torch.no_grad()
    def get_context_embeddings(self, embeddings: np.ndarray) -> np.ndarray:
        t = torch.tensor(embeddings, dtype=torch.float32).unsqueeze(0).to(self.device)
        ctx = self.model.encoder(t, [embeddings.shape[0]])  
        return ctx.squeeze(0).cpu().numpy()

    @torch.no_grad()
    def predict(
        self,
        embeddings: np.ndarray,
        threshold:  float = 0.5,
    ) -> List[Dict]:
        t       = torch.tensor(embeddings, dtype=torch.float32).unsqueeze(0).to(self.device)
        lengths = [embeddings.shape[0]]

        ctx    = self.model.encoder(t, lengths)       
        logits = self.model.classifier1(ctx)
        pres   = self.model.classifier2(ctx)          

        T = embeddings.shape[0]
        results = []

        for t_idx in range(T):
            pred: Dict = {"adaptive": {}, "maladaptive": {}}

            for valence, dim in HEAD_ORDER:
                key      = f"{valence}_{dim}"
                logit_t  = logits[key][0, t_idx]             
                local_idx = logit_t.argmax().item()
                pred[valence][dim] = _INV_HEAD[valence][dim][local_idx]

            pred["switch"]      = logits["switch"][0, t_idx, 0].sigmoid().item() >= threshold
            pred["escalation"]  = logits["escalation"][0, t_idx, 0].sigmoid().item() >= threshold
            pred["ada_presence"] = float(pres[0, t_idx, 0].item())
            pred["mal_presence"] = float(pres[0, t_idx, 1].item())

            results.append(pred)

        return results

    @torch.no_grad()
    def enrich_timeline(
        self,
        timeline_embeddings: np.ndarray,      
    ) -> np.ndarray:                          
        ctx = self.get_context_embeddings(timeline_embeddings)
        return np.concatenate([timeline_embeddings, ctx], axis=1)


# ─────────────────────────────────────────────────────────────────────────────
# 11. Quick-start  (python timeline_model.py)
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import random
    from torch.utils.data import random_split
    from data_structure import load_all_timelines

    # ── Config ────────────────────────────────────────────────────────────
    DATA_DIR     = "../../../data/train/"
    DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
    SEED         = 42
    BATCH_SIZE   = 8        

    INPUT_DIM    = 1357
    LSTM_HIDDEN  = 512      
    LSTM_LAYERS  = 2
    PROJ_DIM     = 512      

    random.seed(SEED)
    torch.manual_seed(SEED)

    # ── Data ──────────────────────────────────────────────────────────────
    timelines = load_all_timelines(DATA_DIR)

    full_ds = TimelineDataset(timelines, skip_no_embedding=True)

    n_train = int(0.8 * len(full_ds))
    n_val   = len(full_ds) - n_train
    train_ds, val_ds = random_split(
        full_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED),
    )

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=TimelineDataset.collate,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=TimelineDataset.collate,
    )

    # ── Model ─────────────────────────────────────────────────────────────
    model = TimelineModel(
        input_dim    = INPUT_DIM,
        lstm_hidden  = LSTM_HIDDEN,
        lstm_layers  = LSTM_LAYERS,
        lstm_dropout = 0.3,
        clf_hidden   = 256,
        proj_dim     = PROJ_DIM,
        reg_hidden   = 128,
    )
    print(f"[Model] Total params: {model.total_params():,}")

    # ── Stage 1 ───────────────────────────────────────────────────────────
    trainer = TimelineModelTrainer(
        model         = model,
        device        = DEVICE,
        lr_stage1     = 1e-3,
        lr_stage2     = 5e-4,
        dim_ce_weight = 1.0,
        se_weight     = 2.0,
        pos_weight    = 5.0,
    )
    # Reduced default epochs to 30 to help prevent early overfitting
    trainer.train_stage1(train_loader, val_loader, epochs=30, save_path="timeline_model_s1.pt")

    # ── Stage 2 ───────────────────────────────────────────────────────────
    model.load_state_dict(torch.load("timeline_model_s1.pt", map_location=DEVICE))
    trainer.train_stage2(train_loader, val_loader, epochs=30, save_path="timeline_model_s2.pt")

    # ── Inference demo ────────────────────────────────────────────────────
    model.load_state_dict(torch.load("timeline_model_s2.pt", map_location=DEVICE))
    infer = TimelineInference(model, device=DEVICE)

    tl = timelines[0]
    emb_matrix = np.stack([p.post_embedding for p in tl.posts if p.post_embedding is not None])
    predictions = infer.predict(emb_matrix)

    print(f"\nTimeline {tl.timeline_id!r}: {len(predictions)} post predictions")
    for i, pred in enumerate(predictions[:3]):
        print(f"  Post {i}: switch={pred['switch']}  esc={pred['escalation']}"
              f"  ada_pres={pred['ada_presence']:.2f}  mal_pres={pred['mal_presence']:.2f}")
        print(f"    adaptive:    {pred['adaptive']}")
        print(f"    maladaptive: {pred['maladaptive']}")

    # ── Enrich embeddings for downstream use ──────────────────────────────
    enriched = infer.enrich_timeline(emb_matrix)
    print(f"\nOriginal embedding dim : {emb_matrix.shape[1]}")
    print(f"Enriched embedding dim : {enriched.shape[1]}  (+ {2*LSTM_HIDDEN} context dims)")


Loaded 30 timelines from: ../../../data/train/
  Timelines             : 30
  Total posts           : 373
  Annotated posts       : 236
  Switch posts          : 78
  Escalation posts      : 75
  Both (S+E) posts      : 27
  Adaptive subelements  : 472
  Maladaptive subelements: 615
[TimelineDataset] 30 timelines loaded  (0 skipped).
[Model] Total params: 11,615,088
[TimelineModel] All parameters unfrozen.
[Stage 1] Trainable params: 11,615,088
[S1] Epoch 001/30 | train total=7.0166  ce_mean=1.1985  switch=1.4341  escalation=1.4749 || val total=6.7821  ce_mean=1.1090  switch=1.4190  escalation=1.4175  ✓ saved
[S1] Epoch 002/30 | train total=6.1084  ce_mean=1.0720  switch=1.3949  escalation=1.1233 || val total=8.1447  ce_mean=0.9939  switch=2.1242  escalation=1.4512
[S1] Epoch 003/30 | train total=6.6646  ce_mean=0.9740  switch=1.4955  escalation=1.3498 || val total=6.6231  ce_mean=0.9273  switch=1.3447  escalation=1.5032  ✓ saved
[S1] Epoch 004/30 | train total=8.3057  ce_mean=0.9100 

In [12]:
import os
import json
import numpy as np
import torch
from data_structure import load_all_timelines
# from timeline_model import TimelineModel, TimelineInference

# --- 1. Configuration (Replaces argparse) ---
class Config:
    test_dir = "../../../data/val/"          # PATH TO YOUR TEST DATASET
    model_path = "./timeline_model_s2.pt"     # PATH TO YOUR TRAINED STAGE 2 WEIGHTS
    output_dir = "./pred_result/"             # DIRECTORY TO SAVE JSONS
    threshold = 0.5                           # Sigmoid threshold for Switch/Escalation
    
    # Model Architecture (Must match your training parameters exactly)
    input_dim = 1357
    lstm_hidden = 512
    lstm_layers = 2
    clf_hidden = 256
    proj_dim = 512
    reg_hidden = 128

args = Config()

# --- 2. Inference Logic ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load Test Data
print(f"Loading test timelines from {args.test_dir}...")
test_timelines = load_all_timelines(args.test_dir)

# Initialize Model and Inference Wrapper
print(f"Loading model weights from {args.model_path}...")
model = TimelineModel(
    input_dim=args.input_dim,
    lstm_hidden=args.lstm_hidden,
    lstm_layers=args.lstm_layers,
    lstm_dropout=0.0, # Dropout not needed for inference
    clf_hidden=args.clf_hidden,
    proj_dim=args.proj_dim,
    reg_hidden=args.reg_hidden,
)

# Load the stage 2 weights
model.load_state_dict(torch.load(args.model_path, map_location=device))
infer = TimelineInference(model, device=device)

task1_predictions = []
task2_predictions = []

print("Generating predictions...")
for tl in test_timelines:
    # Extract valid posts (those that successfully passed through PostEmbedder)
    valid_posts = [p for p in tl.posts if p.post_embedding is not None]
    if not valid_posts:
        continue

    # Stack embeddings: Shape (T, D)
    emb_matrix = np.stack([p.post_embedding for p in valid_posts])
    
    # Run Inference
    preds = infer.predict(emb_matrix, threshold=args.threshold)

    # Format output for each post
    for post, pred in zip(valid_posts, preds):
        
        # --- TASK 1: Self-States ---
        # Bound presence scores to [1, 5] integers
        ada_presence = max(1, min(5, round(pred["ada_presence"])))
        mal_presence = max(1, min(5, round(pred["mal_presence"])))
        
        ada_state = {"Presence": ada_presence}
        for dim, subelement in pred["adaptive"].items():
            if subelement is not None:
                ada_state[dim] = {"subelement": subelement}
                
        mal_state = {"Presence": mal_presence}
        for dim, subelement in pred["maladaptive"].items():
            if subelement is not None:
                mal_state[dim] = {"subelement": subelement}

        task1_predictions.append({
            "timeline_id": tl.timeline_id,
            "post_id": post.post_id,
            "adaptive-state": ada_state,
            "maladaptive-state": mal_state
        })

        # --- TASK 2: Switch & Escalation ---
        task2_predictions.append({
            "timeline_id": tl.timeline_id,
            "post_id": post.post_id,
            "Switch": "S" if pred["switch"] else "0",
            "Escalation": "E" if pred["escalation"] else "0"
        })

# --- 3. Save JSON Files ---
os.makedirs(args.output_dir, exist_ok=True)

task1_out_path = os.path.join(args.output_dir, "task1_pred_val.json")
with open(task1_out_path, "w", encoding="utf-8") as f:
    json.dump(task1_predictions, f, indent=4)
print(f"Task 1 predictions saved to: {task1_out_path} ({len(task1_predictions)} posts)")

task2_out_path = os.path.join(args.output_dir, "task2_pred_val.json")
with open(task2_out_path, "w", encoding="utf-8") as f:
    json.dump(task2_predictions, f, indent=4)
print(f"Task 2 predictions saved to: {task2_out_path} ({len(task2_predictions)} posts)")

Using device: cuda
Loading test timelines from ../../../data/val/...

Loaded 2 timelines from: ../../../data/val/
  Timelines             : 2
  Total posts           : 36
  Annotated posts       : 21
  Switch posts          : 5
  Escalation posts      : 4
  Both (S+E) posts      : 1
  Adaptive subelements  : 22
  Maladaptive subelements: 49
Loading model weights from ./timeline_model_s2.pt...
Generating predictions...
Task 1 predictions saved to: ./pred_result/task1_pred_val.json (36 posts)
Task 2 predictions saved to: ./pred_result/task2_pred_val.json (36 posts)
